In [4]:
from tensorial import gcnn
import hydra
import omegaconf
import reax
import matplotlib.pyplot as plt

## Checkpoint paths

In [ ]:
CONFIG_PATH = "/home/mattia/Desktop/ml_codes/CAMML/e3response/logs/train/runs/2025-12-08_16-57-18/config.yaml"
CKPT_PATH = "/home/mattia/Desktop/ml_codes/CAMML/e3response/logs/train/runs/2025-12-08_16-57-18/checkpoints/last.ckpt"

## Load the module

In [6]:
cfg = omegaconf.OmegaConf.load(CONFIG_PATH)
module: reax.Module = hydra.utils.instantiate(cfg.model, _convert_="object")

FileNotFoundError: [Errno 2] No such file or directory: '/home/mattia/Desktop/MLcodes/e3response/logs/train/runs/2025-12-08_16-57-18/config.yaml'

## Load the checkpoint

In [ ]:
checkpointing = reax.training.get_default_checkpointing()
ckpt = checkpointing.load(CKPT_PATH)
module.set_parameters(ckpt["parameters"])

In [ ]:
data_module = hydra.utils.instantiate(cfg.data, limit = 100, _convert_="object")

In [ ]:
from e3response import utils 

listeners: list[reax.TrainerListener] = utils.instantiate_listeners(cfg.get("listeners"))

logger: list[reax.Logger] = utils.instantiate_loggers(cfg.get("logger"))

trainer: reax.Trainer = hydra.utils.instantiate(cfg.trainer, listeners=listeners, logger=logger)

## Prepare dataset and padding

In [ ]:
from e3response.data import qm9_nmr

dataset = qm9_nmr.QM9NmrDataset(r_max = cfg["data"]["r_max"], data_dir = cfg["data"]["data_dir"], limit = 100)

In [ ]:
batch_size = 1

padding = gcnn.data.GraphBatcher.calculate_padding(dataset, batch_size=batch_size)


In [ ]:
import logging

logging.getLogger("reax.data.utils").setLevel(logging.ERROR)  #stop warning prints

In [ ]:
import jax


results = {}
a_values = [10]
b = 10

for a in a_values:
    graphs = gcnn.data.GraphLoader(list(dataset)[b:b+5], batch_size=batch_size, padding=padding)
    graphs_iter = iter(graphs)

    try:
        batch = next(graphs_iter)[0]  # batch[0] è il GraphsTuple
    except StopIteration:
        raise ValueError("Dataloader vuoto. Controlla gli indici.")

    # Stampa campo 'external_magnetic_field' dall'input
    if isinstance(batch.globals, dict) and "external_magnetic_field" in batch.globals:
        print(f"\n🔹 Input 'external_magnetic_field' (a = {a}):")
        print(batch.globals["external_magnetic_field"])
    else:
        print(f"\n⚠️ 'external_magnetic_field' non trovato nei globals del grafo in input (a = {a})")

    # Sposta sul device
    batch = jax.device_put(batch, device=jax.devices('gpu')[0])

    # Applica il modello
    output_graph = module._model.apply(module.parameters(), batch[0])

    # Stampa campo 'B_induced_predicted' per nodo dall'output
    if isinstance(output_graph.nodes, dict) and "B_induced_predicted" in output_graph.nodes:
        print(f"\n🔸 Output 'B_induced_predicted' per nodo (a = {a}):")
        print(output_graph.nodes["B_induced_predicted"])
    else:
        print(f"\n⚠️ 'B_induced_predicted' non trovato nei nodi del grafo in output (a = {a})")

    # Valida
    test = trainer.validate(module, dataloaders=graphs)
    rmse = test.logged_metrics['val/nmr_tensors_rmse']
    results[a] = rmse

print("\n✅ RISULTATI:")
print(results)


In [ ]:
results = {}
# a_values = [100000]
# a_values = [10, 100, 1000, 10000]
a_values = [10]
b = 10

for a in a_values:
    graphs = gcnn.data.GraphLoader(list(dataset)[b:b+5], batch_size=batch_size, padding=padding)
    
    # Valida il modello
    test = trainer.validate(module, dataloaders=graphs) #using validate because test gives an error for some reason?
    
    print(test)

    # Estrai e salva il valore di interesse
    rmse = test.logged_metrics['val/nmr_tensors_rmse']
    results[a] = rmse

# Stampa o usa i risultati
print(results)




In [ ]:
import jax

graphs = gcnn.data.GraphLoader(dataset, batch_size=batch_size, padding=padding)

batch_tuple = next(iter(graphs))  # probabilmente una tupla (graph, labels)
graph_batch = batch_tuple[0]      # il vero GraphsTuple

# print(batch_tuple)
print(graph_batch)

dev = jax.devices('gpu')[0]
print(dev)
# Sposta sul device
graph_batch = jax.device_put(graph_batch, device=dev)

module._model.apply(module.parameters(), graph_batch)

In [ ]:
import yaml

# Salva i risultati in un file YAML
with open("nequip_nmr_deriv_rmse_results.yaml", "w") as f:
    yaml.dump(results, f)


In [ ]:
# Plot dei risultati
x = list(results.keys())
y = [results[k] for k in x]

plt.figure(figsize=(8, 5))
plt.plot(x, y, marker='o', linestyle='-', color='steelblue')
plt.xlabel("Numero di grafi (a)")
plt.ylabel("val/nmr_tensors_rmse")
plt.title("Andamento dell'RMSE al variare del numero di grafi")
plt.grid(True)
plt.xscale('log')  
plt.tight_layout()
plt.show()


In [ ]:
import yaml
import matplotlib.pyplot as plt

# Carica i risultati dal primo file
with open("nequip_nmr_rmse_results_100mol.yaml", "r") as f:
    rmse_results_100 = yaml.safe_load(f)

# Carica i risultati dal secondo file
with open("nequip_nmr_deriv_rmse_results_100mol.yaml", "r") as f:
    deriv_rmse_results_100 = yaml.safe_load(f)

# Carica i risultati dal secondo file
with open("nequip_nmr_rmse_results_200mol.yaml", "r") as f:
    rmse_results_200 = yaml.safe_load(f)

# Carica i risultati dal secondo file
with open("nequip_nmr_deriv_rmse_results_200mol.yaml", "r") as f:
    deriv_rmse_results_200 = yaml.safe_load(f)

# Carica i risultati dal secondo file
with open("nequip_nmr_rmse_results_500mol.yaml", "r") as f:
    rmse_results_500 = yaml.safe_load(f)

# Carica i risultati dal secondo file
with open("nequip_nmr_deriv_rmse_results_500mol.yaml", "r") as f:
    deriv_rmse_results_500 = yaml.safe_load(f)

# Ordina le chiavi per coerenza nel plot
x1 = sorted(rmse_results_100.keys())
y1 = [rmse_results_100[k] for k in x1]

x2 = sorted(deriv_rmse_results_100.keys())
y2 = [deriv_rmse_results_100[k] for k in x2]

x3 = sorted(rmse_results_200.keys())
y3 = [rmse_results_200[k] for k in x3]

x4 = sorted(deriv_rmse_results_200.keys())
y4 = [deriv_rmse_results_200[k] for k in x4]

x5 = sorted(rmse_results_500.keys())
y5 = [rmse_results_500[k] for k in x5]

x6 = sorted(deriv_rmse_results_500.keys())
y6 = [deriv_rmse_results_500[k] for k in x6]

# Plot comparativo
plt.figure(figsize=(9, 5))
plt.plot(x1, y1, marker='o', linestyle='--', label='Standard model (100mol)', color="#14b837")
plt.plot(x2, y2, marker='s', linestyle='-', label='Diff. learning model (100mol)', color='#14b837')
plt.plot(x3, y3, marker='o', linestyle='--', label='Standard model (200mol)', color="#1B59E0")
plt.plot(x4, y4, marker='s', linestyle='-', label='Diff. learning model (200mol)', color="#1B59E0")
plt.plot(x5, y5, marker='o', linestyle='--', label='Standard model (500mol)', color="#E72121")
plt.plot(x6, y6, marker='s', linestyle='-', label='Diff. learning model (500mol)', color="#E72121")


plt.xlabel("N test samples")
plt.ylabel("RMSE")
# plt.title("Confronto RMSE standard vs derivato")
plt.grid(True)
plt.xscale('log')  
plt.ylim([0, 46])
# plt.yscale('log')
plt.legend()
plt.tight_layout()
plt.show()